# 电影数据关联规则分析 —— Apriori算法

**参考**: 《Python数据挖掘入门与实践》第四章 —— 用亲和性分析方法推荐电影

**数据集**: MovieLens 电影评分数据

**参数设置（自行设定）**:
- **最小支持度 (min_support)**: `40` — 一部电影至少被40个用户好评才纳入分析（约占200个抽样用户的20%）
- **最小置信度 (min_confidence)**: `0.70` — 规则的可信度需达到70%以上才认定为强关联规则

> 说明：min_support 设为40时，只有"大众爆款"电影能入围，用户群体高度重叠导致置信度全是100%。降低到20后，更多"有分歧"的组合能够入围，置信度将呈现70%~100%的自然分布。

## 1. 导入库与加载数据

In [1]:
import pandas as pd
from collections import defaultdict
from operator import itemgetter
import sys
import os

# 数据路径
data_dir = os.path.join(os.getcwd(), "Data", "movie")
ratings_file = os.path.join(data_dir, "ratings.csv")
movies_file = os.path.join(data_dir, "movies.csv")

print("数据目录:", data_dir)

数据目录: c:\my_code\learning-notes\机器学习(作业)\实验九\Chapter4_movie\Data\movie


In [2]:
# 加载评分数据
all_ratings = pd.read_csv(ratings_file)
print(f"评分数据总数: {len(all_ratings)} 条")
print(f"包含列: {list(all_ratings.columns)}")
all_ratings.head()

评分数据总数: 100004 条
包含列: ['userId', 'movieId', 'rating', 'timestamp']


,userId,movieId,rating,timestamp
0,1,31,2.5,1260759144
1,1,1029,3.0,1260759179
2,1,1061,3.0,1260759182
3,1,1129,2.0,1260759185
4,1,1172,4.0,1260759205


In [3]:
# 加载电影信息数据
movie_name_data = pd.read_csv(movies_file)
print(f"电影总数: {len(movie_name_data)} 部")
movie_name_data.head()

电影总数: 9125 部


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


## 2. 数据预处理

In [4]:
# 标记好评：评分 > 3 表示用户喜欢该电影（Favorable = True）
all_ratings["Favorable"] = all_ratings["rating"] > 3

print(f"好评率: {all_ratings['Favorable'].mean():.2%}")
all_ratings[all_ratings["Favorable"]].head()

好评率: 62.10%


,userId,movieId,rating,timestamp,Favorable
4,1,1172,4.0,1260759205,True
8,1,1339,3.5,1260759125,True
12,1,1953,4.0,1260759191,True
13,1,2105,4.0,1260759139,True
20,2,10,4.0,835355493,True


In [5]:
# 抽样用户以控制计算规模（取前200个用户的数据）
# 可根据电脑性能调整用户数量
ratings = all_ratings[all_ratings['userId'].isin(range(200))]
print(f"抽样后评分数: {len(ratings)} 条")
print(f"抽样后用户数: {ratings['userId'].nunique()} 个")

抽样后评分数: 27425 条
抽样后用户数: 199 个


In [6]:
# 筛选出好评数据
favorable_ratings = ratings[ratings["Favorable"]]
print(f"好评记录数: {len(favorable_ratings)} 条")

# 构建用户-电影好评集合（每个用户对应一个电影ID的集合）
favorable_reviews_by_users = dict(
    (k, frozenset(v.values))
    for k, v in favorable_ratings.groupby("userId")["movieId"]
)
print(f"有至少1条好评的用户数: {len(favorable_reviews_by_users)}")

好评记录数: 17152 条
有至少1条好评的用户数: 199


In [7]:
# 统计每部电影的好评数量
num_favorable_by_movie = ratings[["movieId", "Favorable"]].groupby("movieId").sum()
print("好评最多的5部电影:")
num_favorable_by_movie.sort_values("Favorable", ascending=False).head()

好评最多的5部电影:


,Favorable
movieId,
296,88
318,84
356,84
593,72
527,70


## 3. Apriori算法 —— 寻找频繁项集

In [8]:
def find_frequent_itemsets(favorable_reviews_by_users, k_1_itemsets, min_support):
    """
    从 k-1 项频繁项集生成 k 项频繁项集
    
    参数:
        favorable_reviews_by_users: 用户-好评电影集合字典
        k_1_itemsets: 上一轮找到的 (k-1) 项频繁项集
        min_support: 最小支持度阈值
    
    返回:
        k项频繁项集及其支持度的字典
    """
    counts = defaultdict(int)
    for user, reviews in favorable_reviews_by_users.items():
        for itemset in k_1_itemsets:
            if itemset.issubset(reviews):
                for other_reviewed_movie in reviews - itemset:
                    current_superset = itemset | frozenset((other_reviewed_movie,))
                    counts[current_superset] += 1
    return dict([(itemset, freq) for itemset, freq in counts.items() if freq >= min_support])

In [ ]:
# ============ 自行设定参数 ============
MIN_SUPPORT = 50   
MIN_CONFIDENCE = 0.70 # 最小置信度
print(f"参数设置:")
print(f"  最小支持度: {MIN_SUPPORT}")
print(f"  最小置信度: {MIN_CONFIDENCE}")
print("-" * 50)

frequent_itemsets = {}

# 第1步：找出所有1项频繁项集（好评数 > min_support 的电影）
frequent_itemsets[1] = dict(
    (frozenset((movie_id,)), row["Favorable"])
    for movie_id, row in num_favorable_by_movie.iterrows()
    if row["Favorable"] > MIN_SUPPORT
)

print(f"1项频繁项集（好评数 > {MIN_SUPPORT}）: {len(frequent_itemsets[1])} 部电影")

# 第2步：逐层搜索 k=2,3,4,... 的频繁项集
for k in range(2, 20):
    cur_frequent_itemsets = find_frequent_itemsets(
        favorable_reviews_by_users,
        frequent_itemsets[k - 1],
        MIN_SUPPORT
    )
    if len(cur_frequent_itemsets) == 0:
        print(f"未找到 {k} 项频繁项集，Apriori搜索终止")
        break
    else:
        print(f"找到 {len(cur_frequent_itemsets)} 个 {k} 项频繁项集")
        sys.stdout.flush()
        frequent_itemsets[k] = cur_frequent_itemsets

# 删除1项集（单项无法形成关联规则）
del frequent_itemsets[1]

total = sum(len(v) for v in frequent_itemsets.values())
print(f"\n共找到 {total} 个频繁项集（从2项到{max(frequent_itemsets.keys())}项）")

参数设置:
  最小支持度: 50
  最小置信度: 0.6
--------------------------------------------------
1项频繁项集（好评数 > 50）: 21 部电影
找到 157 个 2 项频繁项集
找到 590 个 3 项频繁项集
找到 1250 个 4 项频繁项集
找到 1596 个 5 项频繁项集
找到 1279 个 6 项频繁项集
找到 650 个 7 项频繁项集
找到 199 个 8 项频繁项集
找到 32 个 9 项频繁项集
找到 2 个 10 项频繁项集
未找到 11 项频繁项集，Apriori搜索终止

共找到 5755 个频繁项集（从2项到10项）


## 4. 生成关联规则并计算置信度

In [18]:
# 从频繁项集生成候选关联规则
candidate_rules = []
for itemset_length, itemset_counts in frequent_itemsets.items():
    for itemset in itemset_counts.keys():
        for conclusion in itemset:
            premise = itemset - set((conclusion,))
            candidate_rules.append((premise, conclusion))

print(f"候选规则总数: {len(candidate_rules)} 条")

候选规则总数: 29188 条


In [19]:
# 计算每条候选规则的置信度
correct_counts = defaultdict(int)
incorrect_counts = defaultdict(int)

for user, reviews in favorable_reviews_by_users.items():
    for candidate_rule in candidate_rules:
        premise, conclusion = candidate_rule
        if premise.issubset(reviews):
            if conclusion in reviews:
                correct_counts[candidate_rule] += 1
            else:
                incorrect_counts[candidate_rule] += 1

rule_confidence = {
    candidate_rule:
    correct_counts[candidate_rule] / float(correct_counts[candidate_rule] + incorrect_counts[candidate_rule])
    for candidate_rule in candidate_rules
}
print("置信度计算完成")

置信度计算完成


In [20]:
# ============ 筛选强关联规则 ============
rule_confidence = {
    rule: confidence
    for rule, confidence in rule_confidence.items()
    if confidence > MIN_CONFIDENCE
}
print(f"满足最小置信度 {MIN_CONFIDENCE} 的强关联规则: {len(rule_confidence)} 条")

满足最小置信度 0.6 的强关联规则: 27696 条


In [21]:
# 按置信度降序排列
sorted_confidence = sorted(
    rule_confidence.items(), key=itemgetter(1), reverse=True
)

## 5. 结果展示 —— 强关联规则（含电影名）

In [22]:
# 根据 movieId 获取电影名
def get_movie_name(movie_id):
    title_object = movie_name_data[movie_name_data["movieId"] == movie_id]["title"]
    if len(title_object) > 0:
        return title_object.values[0]
    return f"Unknown (ID:{movie_id})"

# 根据 movieId 获取电影类型
def get_movie_genres(movie_id):
    genres_object = movie_name_data[movie_name_data["movieId"] == movie_id]["genres"]
    if len(genres_object) > 0:
        return genres_object.values[0]
    return "Unknown"

In [15]:
# 展示前20条强关联规则
top_n = 20
print(f"{'='*80}")
print(f"{'Top ' + str(top_n) + ' 强关联规则（最小支持度={}, 最小置信度={}）'.center(70)}".format(MIN_SUPPORT, MIN_CONFIDENCE))
print(f"{'='*80}")

for index in range(min(top_n, len(sorted_confidence))):
    (premise, conclusion) = sorted_confidence[index][0]
    confidence = sorted_confidence[index][1]
    
    premise_names = ", ".join(get_movie_name(mid) for mid in premise)
    conclusion_name = get_movie_name(conclusion)
    
    # 获取前件中第一部电影的类型信息
    first_movie_id = list(premise)[0]
    conclusion_genres = get_movie_genres(conclusion)
    
    print(f"\n规则 #{index + 1}")
    print(f"  如果用户喜欢:")
    for mid in premise:
        print(f"    • {get_movie_name(mid)} ({get_movie_genres(mid)})")
    print(f"  那么他也可能喜欢:")
    print(f"    ★ {conclusion_name} ({conclusion_genres})")
    print(f"  置信度: {confidence:.2%}")

Top 20                       强关联规则（最小支持度=50, 最小置信度=0.7）                      

规则 #1
  如果用户喜欢:
    • Star Wars: Episode VI - Return of the Jedi (1983) (Action|Adventure|Sci-Fi)
    • Terminator 2: Judgment Day (1991) (Action|Sci-Fi)
  那么他也可能喜欢:
    ★ Star Wars: Episode V - The Empire Strikes Back (1980) (Action|Adventure|Sci-Fi)
  置信度: 100.00%

规则 #2
  如果用户喜欢:
    • Pulp Fiction (1994) (Comedy|Crime|Drama|Thriller)
    • Star Wars: Episode VI - Return of the Jedi (1983) (Action|Adventure|Sci-Fi)
    • Terminator 2: Judgment Day (1991) (Action|Sci-Fi)
  那么他也可能喜欢:
    ★ Star Wars: Episode V - The Empire Strikes Back (1980) (Action|Adventure|Sci-Fi)
  置信度: 100.00%

规则 #3
  如果用户喜欢:
    • Pulp Fiction (1994) (Comedy|Crime|Drama|Thriller)
    • Jurassic Park (1993) (Action|Adventure|Sci-Fi|Thriller)
    • Star Wars: Episode V - The Empire Strikes Back (1980) (Action|Adventure|Sci-Fi)
  那么他也可能喜欢:
    ★ Star Wars: Episode IV - A New Hope (1977) (Action|Adventure|Sci-Fi)
  置信度: 100.00%

规则 #4
 

## 6. 汇总分析

In [16]:
# 统计分析
print("=" * 60)
print("关联规则分析汇总")
print("=" * 60)
print(f"数据规模: {len(all_ratings)} 条评分, {movie_name_data['movieId'].nunique()} 部电影")
print(f"抽样用户: {ratings['userId'].nunique()} 个")
print(f"最小支持度: {MIN_SUPPORT}")
print(f"最小置信度: {MIN_CONFIDENCE}")
print(f"频繁项集总数: {total} 个")
print(f"强关联规则数: {len(rule_confidence)} 条")

# 统计前件中电影出现频率
premise_movie_counter = defaultdict(int)
for (premise, conclusion), conf in sorted_confidence:
    for mid in premise:
        premise_movie_counter[mid] += 1

print("\n频繁出现在规则前件中的电影 Top 10:")
for mid, count in sorted(premise_movie_counter.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(f"  {get_movie_name(mid)}: 出现 {count} 次")

关联规则分析汇总
数据规模: 100004 条评分, 9125 部电影
抽样用户: 199 个
最小支持度: 50
最小置信度: 0.7
频繁项集总数: 5755 个
强关联规则数: 24789 条

频繁出现在规则前件中的电影 Top 10:
  Pulp Fiction (1994): 出现 10736 次
  Back to the Future (1985): 出现 10625 次
  Shawshank Redemption, The (1994): 出现 9990 次
  Star Wars: Episode IV - A New Hope (1977): 出现 9535 次
  Matrix, The (1999): 出现 9035 次
  Raiders of the Lost Ark (Indiana Jones and the Raiders of the Lost Ark) (1981): 出现 8664 次
  American Beauty (1999): 出现 8516 次
  Star Wars: Episode V - The Empire Strikes Back (1980): 出现 8179 次
  Star Wars: Episode VI - Return of the Jedi (1983): 出现 8115 次
  Fight Club (1999): 出现 7661 次


## 7. 结论

通过 Apriori 算法对 MovieLens 电影评分数据进行关联规则挖掘，我们得到了以下结论：

1. **算法核心思想**: Apriori 算法基于"先验原理"——如果一个项集是频繁的，那么它的所有子集也是频繁的。通过逐层搜索的方式，从1项频繁项集出发，逐步生成更高阶的频繁项集。

2. **参数选择的含义**:
   - **支持度 (Support)**: 表示某电影集合在所有用户中同时出现的比例。设最小支持度为40，意味着至少40个用户同时喜欢这些电影，该组合才被纳入分析。
   - **置信度 (Confidence)**: 表示在前件出现的条件下，后件也出现的概率。设最小置信度为0.85，意味着规则的可靠性需达85%以上。

3. **强关联规则的意义**: 高置信度的规则揭示了用户观影偏好的内在关联，例如喜欢某系列电影的观众也倾向于喜欢该系列的其他作品。这些规则可用于电影推荐系统。

4. **与《Python数据挖掘入门与实践》第四章的对应**: 本章介绍的是亲和性分析（Affinity Analysis），核心是发现"如果用户购买了X，也可能购买Y"这样的关联关系。Apriori 算法是其中最经典的方法之一。